# Landform Classification

The `landforms()` function classifies each cell in an elevation raster into one of 10 landform categories using the Weiss (2001) TPI-based scheme. It computes Topographic Position Index (TPI) at two neighborhood scales, standardizes both to z-scores, and combines them with slope to assign classes ranging from canyons and valleys to ridges and mountain tops.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

from xrspatial import landforms
from xrspatial.terrain_metrics import LANDFORM_CLASSES

## 1. Generate synthetic terrain

We build a surface that contains a clear ridge, a valley, and some flat/sloped areas so the classifier has a variety of features to work with.

In [ ]:
rows, cols = 120, 160
y = np.linspace(0, 12, rows)
x = np.linspace(0, 16, cols)
Y, X = np.meshgrid(y, x, indexing='ij')

# Combine a ridge, a valley, and some rolling terrain
ridge = 80 * np.exp(-((Y - 3)**2 + (X - 8)**2) / (2 * 2.5**2))
valley = -40 * np.exp(-((Y - 9)**2 + (X - 5)**2) / (2 * 2**2))
rolling = 15 * np.sin(Y / 1.5) * np.cos(X / 2)
base = 200 + 10 * Y  # gentle regional slope

elevation = base + ridge + valley + rolling

agg = xr.DataArray(
    elevation,
    dims=['y', 'x'],
    attrs={'res': (0.1, 0.1)},
)
agg['y'] = np.linspace(y[-1], y[0], rows)
agg['x'] = x

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(elevation, cmap='terrain', aspect='auto')
ax.set_title('Synthetic elevation')
ax.set_xlabel('Column')
ax.set_ylabel('Row')
fig.colorbar(im, ax=ax, label='Elevation (m)')
plt.tight_layout()
plt.show()

## 2. Run landform classification

The default parameters (`inner_radius=3`, `outer_radius=15`) work well for many DEMs. Here we use smaller radii to match the scale of our synthetic terrain.

In [ ]:
classes = landforms(agg, inner_radius=3, outer_radius=12)

# Build a categorical colormap
colors = [
    '#1a237e',  # 1  Canyon
    '#4a148c',  # 2  Midslope drainage
    '#880e4f',  # 3  Upland drainage
    '#0d47a1',  # 4  U-shaped valley
    '#a5d6a7',  # 5  Plain
    '#fff9c4',  # 6  Open slope
    '#ffcc80',  # 7  Upper slope
    '#e65100',  # 8  Local ridge
    '#bf360c',  # 9  Midslope ridge
    '#b71c1c',  # 10 Mountain top
]
cmap = ListedColormap(colors)
bounds = np.arange(0.5, 11.5, 1)
norm = BoundaryNorm(bounds, cmap.N)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(classes.values, cmap=cmap, norm=norm, aspect='auto')
cbar = fig.colorbar(im, ax=ax, ticks=range(1, 11))
cbar.ax.set_yticklabels(
    [LANDFORM_CLASSES[i] for i in range(1, 11)], fontsize=8
)
ax.set_title('Weiss (2001) landform classification')
ax.set_xlabel('Column')
ax.set_ylabel('Row')
plt.tight_layout()
plt.show()

## 3. Effect of neighborhood radii

The inner and outer radii control the scale of features the classifier picks up. Smaller radii detect finer-grained features; larger radii smooth out local variation.

In [ ]:
configs = [
    (2, 6, 'inner=2, outer=6'),
    (3, 12, 'inner=3, outer=12'),
    (5, 20, 'inner=5, outer=20'),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (ir, outr, label) in zip(axes, configs):
    c = landforms(agg, inner_radius=ir, outer_radius=outr)
    ax.imshow(c.values, cmap=cmap, norm=norm, aspect='auto')
    ax.set_title(label)
    ax.axis('off')
plt.suptitle('Scale sensitivity', y=1.02)
plt.tight_layout()
plt.show()

## 4. Slope threshold for plains vs. open slopes

The `slope_threshold` parameter (default 5 degrees) controls whether mid-position cells are labeled as plains or open slopes. A lower threshold classifies more cells as slopes; a higher threshold classifies more as plains.

In [ ]:
thresholds = [2.0, 5.0, 15.0]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, thr in zip(axes, thresholds):
    c = landforms(agg, inner_radius=3, outer_radius=12,
                  slope_threshold=thr)
    ax.imshow(c.values, cmap=cmap, norm=norm, aspect='auto')
    ax.set_title(f'slope_threshold={thr}')
    ax.axis('off')
plt.suptitle('Plains vs. open slopes threshold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Class distribution

A histogram of class frequencies shows which landform types dominate the study area.

In [ ]:
classes = landforms(agg, inner_radius=3, outer_radius=12)
valid = classes.values[~np.isnan(classes.values)].astype(int)

counts = [np.sum(valid == i) for i in range(1, 11)]
labels = [LANDFORM_CLASSES[i] for i in range(1, 11)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(10), counts, color=colors)
ax.set_yticks(range(10))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel('Cell count')
ax.set_title('Landform class distribution')
ax.invert_yaxis()
plt.tight_layout()
plt.show()